# Myproject — Train 100 food classes on Kaggle

This experiment preserves the first **50 class IDs** (20 Food-101 + 30 Vietnamese), and adds 50 Food-101 labels. EfficientNet-B0, per-class validation, best checkpoint by macro-F1, held-out test results, and ONNX export are included.

**Data required:** add a Vietnamese Foods dataset with `Images/{Train,Validate,Test}/<class>` folders, plus **Food-101** (`food-101/images` and `food-101/meta` in its parent root). If Food-101 is not attached, enable Kaggle Internet for automatic download (~5 GB).

**Important:** Training is not the same as deployment. The live 12-class service remains unchanged until its ONNX model, class list and nutrition lookup are updated together.


In [ ]:
from pathlib import Path
import subprocess, json, sys, shutil

VN_ROOT = Path('/kaggle/input/vietnamese-foods/Images')
FOOD_ROOT = Path('/kaggle/working/data')  # parent folder containing 'food-101/'
OUTPUT = Path('/kaggle/working/food100')
REPO = Path('/kaggle/working/Myproject')

print('Vietnamese Foods ready:', (VN_ROOT / 'Train').exists())
print('Food-101 ready:', (FOOD_ROOT / 'food-101' / 'meta').exists())
print('GPU:', (subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip() if shutil.which('nvidia-smi') else '') or 'Not detected')


### 1. Load the training code

Enable Kaggle Internet for this cell. Alternatively add a Kaggle dataset containing `ml/train_food_100.py` and change `TRAIN_SCRIPT` to that file. The script lives on an isolated feature branch of Myproject.


In [ ]:
if not (REPO / 'ml' / 'train_food_100.py').exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'feature/food100-training',
                    'https://github.com/Tl4826/Myproject.git', str(REPO)], check=True)
TRAIN_SCRIPT = REPO / 'ml' / 'train_food_100.py'
assert TRAIN_SCRIPT.is_file(), 'Set TRAIN_SCRIPT to the uploaded training script'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'scikit-learn', 'onnx', 'onnxruntime', 'pillow'], check=True)
print('Ready:', TRAIN_SCRIPT)


### 2. Optional warm-start from the old 50-class model

Upload the **real 50-class PyTorch checkpoint** (`best.pt`/`.pth`) as a Kaggle dataset and fill `OLD_CHECKPOINT`. If its checkpoint does not embed `classes`, also supply the exact 50-class `classes.json` from the same run. Without it, the script will train a fresh 100-class head from ImageNet-pretrained EfficientNet-B0 and learn all 100 classes. **Do not use the deployed 12-class model as your 50-class checkpoint.**


In [ ]:
OLD_CHECKPOINT = None  # Example: Path('/kaggle/input/my-food50/best.pt')
OLD_CLASSES = None     # Example: Path('/kaggle/input/my-food50/classes.json')

cmd = [sys.executable, str(TRAIN_SCRIPT), '--vn-root', str(VN_ROOT),
       '--food-root', str(FOOD_ROOT), '--out', str(OUTPUT),
       '--epochs-head', '2', '--epochs-full', '8',
       '--batch-size', '32', '--max-food-train', '350']
if not (FOOD_ROOT / 'food-101' / 'meta').exists():
    cmd.append('--download-food101')  # Needs Kaggle Internet enabled
if OLD_CHECKPOINT is not None:
    cmd.extend(['--old-checkpoint', str(OLD_CHECKPOINT)])
if OLD_CLASSES is not None:
    cmd.extend(['--old-classes', str(OLD_CLASSES)])
assert (VN_ROOT / 'Train').exists(), 'Attach Vietnamese Foods and fix VN_ROOT'
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


### 3. Check results and package model

After training, inspect **test macro-F1**, per-class report and potential confusion between related dishes. The ONNX + classes.json must always be deployed together, with nutrition labels added for all 100 classes.


In [ ]:
metrics = json.loads((OUTPUT / 'test_metrics.json').read_text())
labels = json.loads((OUTPUT / 'classes.json').read_text())['classes']
assert len(labels) == len(set(labels)) == 100
print(json.dumps(metrics, indent=2))
print('ONNX ready:', (OUTPUT / 'food_classifier.onnx').is_file())
archive = shutil.make_archive('/kaggle/working/myproject_food100', 'zip', OUTPUT)
print('Download from Kaggle Output:', archive)
